# StyleGAN2-ADA Animefaces 512 Fresh Stable Run

This notebook starts a clean 512x512 run from scratch. It avoids the two failure modes seen in the previous runs:

- Geometric ADA pipelines such as `bgc` can teach sideways/rotated faces.
- The built-in `color` pipeline includes luminance and hue transforms that can push this dataset toward dark or inverted faces.

The default here is intentionally conservative: `aug=noaug`, `resume=noresume`, a fresh `/content` work directory, fixed `gamma`, and frequent snapshots for early visual checks. Training artifacts stay under `/content` during training, then the whole work directory is zipped to `MyDrive/alibaba` when training exits cleanly.

In [ ]:
#@title StyleGAN2-ADA Animefaces 512 从零稳定训练
REPO_URL = "https://github.com/DavidFeng0718/AnimeGAN.git" #@param {type:"string"}
BRANCH = "stylegan512" #@param {type:"string"}
WORKDIR = "/content/AnimeGAN" #@param {type:"string"}
CONTENT_ROOT = "/content/stylegan2_ada_animefaces_512_fresh_stable" #@param {type:"string"}
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/alibaba" #@param {type:"string"}
KAGGLE_USERNAME = "" #@param {type:"string"}
KAGGLE_KEY = "" #@param {type:"string"}
KIMG = 3000 #@param {type:"integer"}
BATCH_SIZE = 64 #@param {type:"integer"}
GAMMA = 2.5 #@param {type:"number"}
SNAP = 10 #@param {type:"integer"}
CLEAN_CONTENT_ROOT = True #@param {type:"boolean"}

import json
import os
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
from datetime import datetime

def run(cmd, cwd=None, env=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

def q(value):
    return shlex.quote(str(value))

def get_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

def configure_kaggle(username, key):
    kaggle_dir = Path("/content/kaggle")
    kaggle_json = kaggle_dir / "kaggle.json"
    username = (username or "").strip() or get_colab_secret("KAGGLE_USERNAME")
    key = (key or "").strip() or get_colab_secret("KAGGLE_KEY")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    if username and key:
        kaggle_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    elif Path("/content/kaggle.json").exists():
        shutil.copy2("/content/kaggle.json", kaggle_json)
    elif not kaggle_json.exists():
        from google.colab import files
        print("Upload kaggle.json from Kaggle Account settings.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise RuntimeError("kaggle.json was not uploaded and Kaggle credentials are missing.")
        kaggle_json.write_bytes(uploaded["kaggle.json"])
    kaggle_json.chmod(0o600)
    return kaggle_dir

workdir = Path(WORKDIR).resolve()
content_root = Path(CONTENT_ROOT).resolve()
drive_backup_dir = Path(DRIVE_BACKUP_DIR).resolve()

if CLEAN_CONTENT_ROOT and content_root.exists():
    print(f"Removing old clean-run directory: {content_root}")
    shutil.rmtree(content_root)

dataset_zip = content_root / "datasets" / "animefaces_512.zip"
download_dir = content_root / "kaggle_download"
run_root = content_root / "runs"
config_dir = content_root / "configs"
for path in [dataset_zip.parent, download_dir, run_root, config_dir]:
    path.mkdir(parents=True, exist_ok=True)

from google.colab import drive
drive.mount("/content/drive")

run("nvidia-smi || true")

if workdir.exists():
    run(f"git -C {q(workdir)} fetch origin {q(BRANCH)}")
    run(f"git -C {q(workdir)} checkout {q(BRANCH)}")
    run(f"git -C {q(workdir)} pull --ff-only origin {q(BRANCH)}")
else:
    run(f"git clone --branch {q(BRANCH)} --single-branch {q(REPO_URL)} {q(workdir)}")

project_dir = workdir / "StyleGAN2-ADA"
run(f"{q(sys.executable)} -m pip install --upgrade pip")
run(f"{q(sys.executable)} -m pip install -r requirements.txt --upgrade-strategy only-if-needed", cwd=project_dir)
run(f"{q(sys.executable)} -m pip install kaggle --upgrade-strategy only-if-needed")

kaggle_config_dir = configure_kaggle(KAGGLE_USERNAME, KAGGLE_KEY)

base_config = project_dir / "configs" / "animefaces_512.json"
config = json.loads(base_config.read_text(encoding="utf-8"))
config.update({
    "experiment_name": "stylegan2_ada/animefaces_512_fresh_stable",
    "device": "cuda",
    "dataset_zip": str(dataset_zip),
    "run_root": str(run_root),
    "kimg": int(KIMG),
    "batch_size": int(BATCH_SIZE),
    "snap": int(SNAP),
    "aug": "noaug",
    "augpipe": "color",
    "gamma": float(GAMMA),
    "resume": "noresume",
    "mirror": True,
    "allow_tf32": True,
    "nhwc": True,
    "workers": 8,
})

runtime_config = config_dir / "stylegan2_ada_animefaces_512_fresh_stable.json"
runtime_config.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Runtime config:", runtime_config)
print(json.dumps({k: config[k] for k in ["experiment_name", "kimg", "batch_size", "snap", "aug", "augpipe", "gamma", "resume", "mirror", "allow_tf32", "nhwc", "workers"]}, indent=2))

env = os.environ.copy()
env.update({
    "CONFIG": str(runtime_config),
    "DATASET_ZIP": str(dataset_zip),
    "DOWNLOAD_DIR": str(download_dir),
    "RUN_TRAIN": "1",
    "STYLEGAN_DEVICE": "cuda",
    "KAGGLE_CONFIG_DIR": str(kaggle_config_dir),
})

run("bash scripts/prepare_animefaces_512.sh", cwd=project_dir, env=env)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_base = Path("/content") / f"stylegan2_ada_animefaces_512_fresh_stable_{timestamp}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=content_root)
drive_backup_dir.mkdir(parents=True, exist_ok=True)
drive_archive = drive_backup_dir / Path(archive_path).name
shutil.copy2(archive_path, drive_archive)

print("Training complete.")
print("Local archive:", archive_path)
print("Copied archive:", drive_archive)
print("Local runs:", run_root / "stylegan2_ada" / "animefaces_512_fresh_stable")

## Why This Preset

- `resume=noresume`: avoids carrying polluted 604/645/806 weights into a new run.
- `aug=noaug`: avoids `bgc` geometry and `color` luminance/hue transforms. `augpipe` remains `color` only because the upstream CLI requires a valid value when the config schema is loaded; it is not used when `aug=noaug`.
- `gamma=2.5`: stronger than StyleGAN2-ADA auto gamma for 512/batch64, but less aggressive than reusing the old batch8 gamma. This is a compromise between stability and not over-regularizing the discriminator.
- `batch_size=64`: keeps your faster throughput assumption.
- `snap=10`: saves examples/checkpoints every about 40 kimg so collapse is visible early.
- `allow_tf32=true`, `nhwc=true`, `workers=8`: speed settings, not quality settings.

If samples still develop the same black nose/mouth artifact from scratch, inspect the real dataset for repeated masks/occlusions and try `GAMMA=6.5536`. If results are too soft but stable, try `GAMMA=1.0` or `0.8192`.